In [1]:
# Paramétrage de l'environnement de travail et import des packages

In [2]:
import sys
from pathlib import Path

In [8]:
ROOT = Path.cwd().parents[0]

MODEL_DATA = ROOT / "04_model"
PROCESSED_DATA = ROOT / "01_data" / "02_processed"

%load_ext autoreload
%autoreload 2
sys.path.append(str(ROOT / "03_fonctions"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import pandas as pd
import numpy as np
import shap
from fonctions_perso.machine_learning import waterfall_plot
import joblib
from sklearn import set_config

# Interpretation - SHAP

**Import de 'x_train' et du modèle non calibré**

In [19]:
# x_train
x_test = pd.read_parquet(PROCESSED_DATA / "x_test_for_shap.parquet")

# modèle
modele = joblib.load(MODEL_DATA / "fraud_detection_xgb_pas_calibre.joblib")

In [20]:
x_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 927544 entries, 0 to 1852393
Data columns (total 9 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   type_magasin               927544 non-null  string 
 1   montant_transaction        927544 non-null  float64
 2   etat_client                927544 non-null  string 
 3   population_ville_client    927544 non-null  int64  
 4   mois_transaction           927544 non-null  object 
 5   jour_transaction           927544 non-null  object 
 6   heure_transaction          927544 non-null  string 
 7   distance_domicile_magasin  927544 non-null  float64
 8   age_client                 927544 non-null  int32  
dtypes: float64(2), int32(1), int64(1), object(2), string(3)
memory usage: 67.2+ MB


**Initialisation**

In [ ]:
set_config(transform_output="pandas")

# Appliquer toutes les étapes sans le modèle
x_test_proc = modele[:-1].transform(x_test)

# Récuperer le modèle "nu"
xgb = modele.named_steps["model"]


# Initialisation de l'explainer
explainer = shap.TreeExplainer(xgb, x_test_proc)

shap_values = explainer(x_test_proc)

**Feature Importance**

In [ ]:
shap.plots.bar(shap_values, max_display=10)

**Beeswarm Plot**

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

**Waterfall Plot**

In [ ]:
waterfall_plot(
    shap_values,
    x_train_proc,
    89,
    max_display=10
)